# Getting started with InstaNovo-FM

InstaNovo-FM is a self-supervised foundation model for tandem mass spectra. Instead of learning to predict a peptide sequence during pretraining, it reconstructs masked regions of a spectrum and turns every spectrum into a reusable 768-dimensional representation. This notebook loads the published encoder, embeds real benchmark MS/MS spectra, demonstrates spectral-library retrieval, and shows how to run the companion *de novo* sequencer.

The tutorial uses a small, real slice of the public nine-species benchmark (100 annotated MS/MS spectra fetched through the Hugging Face dataset API). It is small enough for Colab but large enough to contain replicate peptide spectra for a meaningful retrieval demonstration. You can replace it with an MGF, mzML, mzXML, CSV, parquet file, or a larger `SpectrumDataFrame` for an analysis of your own.

## Why embed spectra?

The frozen encoder is useful wherever a spectral representation is more helpful than a task-specific predictor. The accompanying manuscript demonstrates that these embeddings can support:

- **Database-free retrieval and rescue.** Search an embedding index of identified reference spectra, then validate close query–anchor pairs with precursor mass, raw-spectrum similarity, and fragment-ion evidence. This is especially interesting for spectra that a conventional search leaves unassigned.
- **Modification and glycan analysis.** Train a small probe on frozen embeddings instead of retraining the encoder. The manuscript reports strong detection of phosphorylation and glycosylation, and resolves coarse N-glycan families.
- **Run-level quality control and classification.** Aggregate embeddings across a run to characterize experimental conditions without needing peptide or protein identifications.
- **De novo sequencing.** Use the separately released encoder–decoder checkpoint when the desired output is a peptide sequence.

These are research workflows, not identification calls by themselves: use held-out data and an appropriate target–decoy or other validation scheme before transferring any biological label.

## Environment

This notebook is designed to run in [Google Colab](https://colab.research.google.com/) from the published PyPI package — no repository checkout is required. In Colab, select **Runtime → Change runtime type → T4 GPU** before running the cells. CPU works for this 100-spectrum tutorial, but a GPU is much faster for a real library or a *de novo* batch.

The next cell installs the published `instanovo-fm==0.1.0` release and displays pip's download progress. Colab preloads NumPy, while InstaNovo-FM installs a pinned NumPy version; therefore the cell restarts the runtime once after installation. After the restart, run that cell again and then continue. Checkpoints are downloaded automatically from the GitHub release and cached under `~/.cache/instanovo-fm/`.

In [ ]:
import importlib.metadata
import importlib.util
import os
from pathlib import Path

PACKAGE_VERSION = "0.1.0"
is_colab = Path("/content").is_dir()
restart_marker = Path("/content") / f".instanovo_fm_{PACKAGE_VERSION}_ready"

# Colab preloads NumPy. Pip can replace it to satisfy the published package's
# reproducible dependency set, but the already-imported NumPy C extension then
# no longer matches its Python files. Install visibly and restart once before
# importing scientific packages. The marker prevents a restart loop.
needs_colab_bootstrap = is_colab and not restart_marker.exists()
try:
    installed_version = importlib.metadata.version("instanovo-fm")
except importlib.metadata.PackageNotFoundError:
    installed_version = None

requirements = []
if installed_version != PACKAGE_VERSION:
    requirements.append(f"instanovo-fm=={PACKAGE_VERSION}")
if importlib.util.find_spec("matplotlib") is None:
    requirements.append("matplotlib>=3.9")

if needs_colab_bootstrap or requirements:
    print("Installing InstaNovo-FM and its dependencies. This may take a few minutes...")
    %pip install --upgrade --progress-bar on instanovo-fm==0.1.0 matplotlib>=3.9

if needs_colab_bootstrap:
    restart_marker.touch()
    print("Installation complete. Restarting the Colab runtime; run this cell once more afterwards.")
    os.kill(os.getpid(), 9)

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from instanovo_fm.data import FoundationalDataProcessor
from instanovo_fm.model.encoder import FoundationModel
from instanovo_fm.utils.spectrum_dataframe import SpectrumDataFrame

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}.")

## Load the published foundation encoder

`instanovo-fm-v0.1.0` is the published 89.5M-parameter Thompson-span / isotope-co-masking model. Calling `describe_pretrained` is a convenient way to record exactly which checkpoint was used in an analysis.

In [ ]:
MODEL_ID = "instanovo-fm-v0.1.0"

pd.Series(FoundationModel.describe_pretrained(MODEL_ID), name=MODEL_ID)

In [ ]:
model, model_config = FoundationModel.from_pretrained(MODEL_ID)
model = model.to(device).eval()

print(f"Loaded {MODEL_ID}: {model.n_layers} layers, {model.dim_model}-dimensional embeddings")

## Load and view an MS/MS spectrum

The tutorial fetches 100 real spectra from `InstaDeepAI/ms_ninespecies_benchmark` using the Hugging Face rows API. This avoids downloading the full benchmark while preserving the original peak arrays and peptide annotations. Labels are used only to evaluate the retrieval demo — the foundation encoder itself does not need them. To use your own data instead, replace this cell with `SpectrumDataFrame.load(...)`; it supports MGF, mzML, mzXML, CSV, and parquet inputs.

In [ ]:
import json
from urllib.request import urlopen

HF_ROWS_URL = (
    "https://datasets-server.huggingface.co/first-rows?"
    "dataset=InstaDeepAI%2Fms_ninespecies_benchmark&config=default&split=test&offset=0&length=100"
)
with urlopen(HF_ROWS_URL) as response:
    rows = [item["row"] for item in json.load(response)["rows"]]

# SpectrumDataFrame gives the same processing interface for remote benchmark
# rows and for a local MGF/mzML/parquet file.
sdf = SpectrumDataFrame.from_pandas(pd.DataFrame(rows), shuffle=False, is_annotated=True)
raw_dataset = sdf.to_dataset(in_memory=True)
print(f"Loaded {len(raw_dataset)} real benchmark spectra.")

preview_columns = ["scan_number", "sequence", "precursor_mz", "precursor_charge", "retention_time"]
sdf.to_pandas()[[column for column in preview_columns if column in sdf.to_pandas().columns]]

In [ ]:
example = raw_dataset[0]
fig, ax = plt.subplots(figsize=(12, 4))
ax.vlines(example["mz_array"], 0, example["intensity_array"], color="#1f77b4", linewidth=1)
ax.set(xlabel="m/z", ylabel="intensity", title=f"MS/MS spectrum: {example.get('sequence', 'unannotated')}")
plt.show()

## Preprocess exactly as the checkpoint expects

A model embedding is only meaningful when the input preprocessing matches training. `FoundationalDataProcessor` retains the most intense peaks, square-root transforms and L2-normalizes intensity, normalizes m/z when required by the checkpoint, and pads the result. We disable masking here because this is inference rather than self-supervised training.

In [ ]:
def make_foundation_processor(config):
    return FoundationalDataProcessor(
        n_peaks=config.get("n_peaks", 200),
        min_mz=config.get("min_mz", 50.0),
        max_mz=config.get("max_mz", 2500.0),
        min_intensity=config.get("min_intensity", 1e-6),
        remove_precursor_tol=0.0,
        use_spectrum_utils=config.get("use_spectrum_utils", False),
        normalize_mz=config.get("normalize_mz", True),
        peak_ordering=config.get("peak_ordering", "sorted"),
        annotated=False,
        masking_strategy="none",
    )

processor = make_foundation_processor(model_config)
processed_dataset = processor.process_dataset(raw_dataset)
loader = DataLoader(processed_dataset, batch_size=min(32, len(processed_dataset)), collate_fn=processor.collate_fn)
batch = next(iter(loader))
spectra = batch["spectra"].to(device)

def embed_dataset(dataset, processor, model, batch_size=32):
    """Embed every processed spectrum in batches."""
    embeddings = []
    for data_batch in DataLoader(dataset, batch_size=batch_size, collate_fn=processor.collate_fn):
        embeddings.append(model.encode_mean_pooled(data_batch["spectra"].to(device)))
    return torch.cat(embeddings)

embeddings = embed_dataset(processed_dataset, processor, model)
print(f"Model input shape: {tuple(spectra.shape)}  (batch, padded peaks, [m/z, intensity])")
print(f"Embedded all {len(embeddings)} spectra.")

## Turn each spectrum into a learned fingerprint

The encoder converts each MS/MS spectrum into 768 numbers: a **spectral fingerprint**. It is not a list of identified ions, a peptide sequence, or a confidence score. Instead, the fingerprint summarizes peak patterns that the model learned during masked-spectrum reconstruction — for example, fragmentation ladders, isotope and neutral-loss relationships, and broader acquisition context.

For the downstream applications in the manuscript, use `encode_mean_pooled`. It summarizes the final representation of all observed peaks into one vector per spectrum. Spectra with similar fingerprints are candidate neighbours for the same peptide, related chemistry, or similar experimental context; the appropriate interpretation depends on the library and task.

In [ ]:
print(f"Embedded {len(embeddings)} spectrum/s as {embeddings.shape[1]}-number fingerprints.")
print(f"Fingerprint length: {embeddings[0].norm():.3f} (1.000 is expected).")
print("A length of 1 is only a normalization convention: it makes fingerprints comparable,")
print("not a measure of spectral quality, identification confidence, or biological relevance.")

## A small real-data retrieval demo

The 100-spectrum benchmark slice contains replicate observations of several peptide sequences. We use spectra at even row indices as a tiny **anchor library**, and spectra at odd row indices as queries. This is a transparent demonstration of the retrieval calculation: a query is counted as recovered when its nearest anchor has the same annotated sequence. It is not a paper benchmark — the split is tiny, not peptide-disjoint, and the labels are used only for this sanity check.

Because the fingerprints have unit length, their dot product is cosine similarity: **1** means identical fingerprints, values nearer **0** mean less similar directions, and **−1** would mean opposite directions. Treat similarity as a *ranking* signal, not an identification confidence or a universal cutoff. In a real library, compare a candidate to the score distribution of known matches and non-matches, then validate it with raw-spectrum and precursor evidence.

In [ ]:
import numpy as np

raw_table = sdf.to_pandas().reset_index(drop=True)
anchor_indices = np.arange(0, len(raw_table), 2)
query_indices = np.arange(1, len(raw_table), 2)
anchor_sequences = raw_table.loc[anchor_indices, "sequence"].astype(str).tolist()
anchor_sequence_set = set(anchor_sequences)
query_indices = np.array([i for i in query_indices if str(raw_table.loc[i, "sequence"]) in anchor_sequence_set])

anchor_embeddings = embeddings[torch.as_tensor(anchor_indices)]
query_embeddings = embeddings[torch.as_tensor(query_indices)]
similarities = query_embeddings @ anchor_embeddings.T
top_scores, top_positions = similarities.max(dim=1)
retrieved_indices = anchor_indices[top_positions.numpy()]
retrieved_sequences = raw_table.loc[retrieved_indices, "sequence"].astype(str).to_numpy()
query_sequences = raw_table.loc[query_indices, "sequence"].astype(str).to_numpy()

retrieval_results = pd.DataFrame({
    "query_row": query_indices,
    "query_sequence": query_sequences,
    "retrieved_anchor_row": retrieved_indices,
    "retrieved_sequence": retrieved_sequences,
    "embedding_cosine": top_scores.numpy(),
})
retrieval_results["sequence_match"] = retrieval_results["query_sequence"] == retrieval_results["retrieved_sequence"]
print(f"Queries with at least one same-sequence anchor: {len(retrieval_results)}")
print(f"Demonstration Recall@1 on this tiny, non-independent split: {retrieval_results['sequence_match'].mean():.1%}")
retrieval_results.sort_values("embedding_cosine", ascending=False).head(10)

### Turn the pattern into a spectral-library search

For a reference library, embed the references once and persist their embeddings with the anchor metadata. For each query batch, calculate the matrix product below and retain the top candidates. In a production workflow, validate candidate label transfers with appropriate controls and evidence from the original spectra; a close embedding alone is insufficient. For larger libraries, replace the matrix product with a FAISS index.

In [ ]:
# reference_embeddings: (n_reference, 768), already L2-normalized
# query_embeddings:     (n_query, 768), already L2-normalized
# reference_metadata:   a DataFrame indexed in the same order as reference_embeddings

def retrieve_top_k(query_embeddings, reference_embeddings, reference_metadata, k=5):
    similarities = query_embeddings @ reference_embeddings.T
    scores, indices = similarities.topk(min(k, len(reference_embeddings)), dim=1)
    rows = []
    for query_id, (query_scores, query_indices) in enumerate(zip(scores, indices, strict=True)):
        for rank, (score, index) in enumerate(zip(query_scores.tolist(), query_indices.tolist(), strict=True), start=1):
            row = {"query_id": query_id, "rank": rank, "embedding_cosine": score}
            row.update(reference_metadata.iloc[index].to_dict())
            rows.append(row)
    return pd.DataFrame(rows)

# Search the real anchor library for the first query.
anchor_metadata = raw_table.iloc[anchor_indices].reset_index(drop=True)[["sequence", "precursor_mz", "precursor_charge"]]
retrieve_top_k(query_embeddings[:1], anchor_embeddings, anchor_metadata, k=5)

## From spectrum embeddings to a run representation

Run-level analysis is just another pooling step. Embed all spectra in a run in batches, then mean-pool and renormalize their vectors. These run representations can drive visualization, outlier detection, or a simple classifier for known technical labels. Keep experiments separate between training and evaluation when assessing generalization.

In [ ]:
import torch.nn.functional as F

def embed_run(dataset, processor, model, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=processor.collate_fn)
    spectrum_embeddings = []
    for batch in loader:
        spectrum_embeddings.append(model.encode_mean_pooled(batch["spectra"].to(device)))
    return torch.cat(spectrum_embeddings)

# The benchmark slice gives a small run-like aggregate; use a complete raw-file
# dataset here for a meaningful run-level representation.
run_spectrum_embeddings = embed_run(processed_dataset, processor, model)
run_embedding = F.normalize(run_spectrum_embeddings.mean(dim=0), dim=0)
print(f"Run representation shape: {tuple(run_embedding.shape)}")

## De novo peptide sequencing

The foundation encoder itself produces embeddings, not amino-acid sequences. The release also contains a downstream encoder–decoder checkpoint initialized from the foundation model and fine-tuned for *de novo* sequencing. The cell below runs beam search on the same small input. Its prediction is a model hypothesis; apply confidence and FDR controls before using it in an analysis.

In [ ]:
from instanovo.inference import BeamSearchDecoder
from instanovo_fm.downstream.de_novo_sequencing.data import DownstreamDeNovoDataProcessor
from instanovo_fm.downstream.de_novo_sequencing.model import DownstreamDeNovo

DENOVO_MODEL_ID = "instanovo-fm-denovo-v0.1.0"
denovo_model, denovo_config = DownstreamDeNovo.from_pretrained(DENOVO_MODEL_ID)
denovo_model = denovo_model.to(device).eval()

denovo_processor = DownstreamDeNovoDataProcessor(
    residue_set=denovo_model.residue_set,
    n_peaks=denovo_config.get("n_peaks", 200),
    min_mz=denovo_config.get("min_mz", 50.0),
    max_mz=denovo_config.get("max_mz", 2500.0),
    min_intensity=denovo_config.get("min_intensity", 1e-6),
    remove_precursor_tol=denovo_config.get("remove_precursor_tol", 2.0),
    use_spectrum_utils=denovo_config.get("use_spectrum_utils", False),
    normalize_mz=denovo_config.get("normalize_mz", True),
    annotated=False,
    return_str=True,
)
denovo_dataset = denovo_processor.process_dataset(raw_dataset)
denovo_batch = next(iter(DataLoader(denovo_dataset, batch_size=len(denovo_dataset), collate_fn=denovo_processor.collate_fn)))

decoder = BeamSearchDecoder(model=denovo_model)
output = decoder.decode(
    spectra=denovo_batch["spectra"].to(device),
    precursors=denovo_batch["precursors"].to(device),
    beam_size=5,
    max_length=denovo_config.get("max_length", 40),
    return_beam=True,
)

sequence_log_probability = output["prediction_log_probability"]
if isinstance(sequence_log_probability, torch.Tensor):
    sequence_log_probability = sequence_log_probability.detach().cpu().numpy()

predicted_tokens = output["predictions"]
predicted_peptides = ["".join(tokens) for tokens in predicted_tokens]
model_confidence = np.clip(np.exp(sequence_log_probability), 0, 1)
denovo_table = pd.DataFrame({
    "spectrum_id": [f"demo_{i:04d}" for i in range(len(predicted_peptides))],
    "predicted_peptide": predicted_peptides,
    "sequence_log_probability": sequence_log_probability,
    "model_confidence": model_confidence,
})

# Winnow consumes the same spectrum table and prediction columns used by InstaNovo's CLI.
winnow_spectra = raw_table.iloc[:len(denovo_table)].copy()
winnow_spectra.insert(0, "spectrum_id", denovo_table["spectrum_id"].to_numpy())
winnow_spectra.to_parquet("instanovo_winnow_spectra.parquet", index=False)
winnow_predictions = {
    "spectrum_id": denovo_table["spectrum_id"],
    "predictions": predicted_peptides,
    "predictions_tokenised": [", ".join(tokens) for tokens in predicted_tokens],
    "log_probs": sequence_log_probability,
}
for beam_idx in range(5):
    winnow_predictions[f"predictions_beam_{beam_idx}"] = ["".join(tokens) for tokens in output[f"predictions_beam_{beam_idx}"]]
    winnow_predictions[f"predictions_log_probability_beam_{beam_idx}"] = output[f"predictions_log_probability_beam_{beam_idx}"]
    winnow_predictions[f"predictions_token_log_probabilities_beam_{beam_idx}"] = [str(values) for values in output[f"predictions_token_log_probabilities_beam_{beam_idx}"]]
pd.DataFrame(winnow_predictions).to_csv("instanovo_winnow_predictions.csv", index=False)

denovo_table

## Calibrate confidence and control FDR with Winnow

The de novo model's log probability is useful for ranking candidates, but it is not a calibrated probability: a score of 0.9 would not necessarily mean a 90% chance that the peptide is correct. [Winnow](https://github.com/instadeepai/winnow) uses a pretrained calibrator and predicted fragment ions to estimate a q-value (an FDR-controlled confidence measure) without requiring ground-truth labels. This is a practical way to retain only peptide-spectrum matches (PSMs) meeting a target error rate.

The cells below use the 100 spectra scored above, download Winnow's calibrator, and query the public Koina service. Set the collision energy and fragmentation type to match your acquisition method before interpreting the results.

In [ ]:
print("Installing/checking Winnow (progress is shown below)...")
%pip install --upgrade-strategy only-if-needed winnow-fdr==2.0.0
print("Winnow is ready.")

In [ ]:
# Return all PSMs from Winnow, then apply the 5% q-value threshold ourselves below.
!winnow predict dataset.spectrum_path_or_directory=instanovo_winnow_spectra.parquet \
    dataset.predictions_path=instanovo_winnow_predictions.csv \
    koina.input_constants.collision_energies=30 \
    koina.input_constants.fragmentation_types=HCD \
    output_folder=winnow_results \
    fdr_control.fdr_threshold=1.0 \
    +calibrator.irt_calibration.train_fraction=0.3

In [ ]:
metadata = pd.read_csv("winnow_results/metadata.csv")
metrics = pd.read_csv("winnow_results/preds_and_fdr_metrics.csv")
winnow_results = metadata.merge(metrics, on="spectrum_id")

target_fdr = 0.05
accepted = winnow_results[winnow_results["psm_q_value"] <= target_fdr].copy()
print(f"Accepted {len(accepted)} of {len(winnow_results)} PSMs at q <= {target_fdr:.0%}.")
display(accepted[["spectrum_id", "predictions", "psm_q_value"]].head(10))

plot_data = winnow_results.sort_values("calibrated_confidence")
ax = plot_data.plot("calibrated_confidence", "psm_q_value", marker=".", legend=False, figsize=(6, 4))
ax.axhline(target_fdr, color="tab:red", linestyle="--", label=f"q = {target_fdr:.0%}")
ax.set(xlabel="Winnow calibrated confidence", ylabel="PSM q-value", title="Confidence calibration and FDR control")
ax.legend()
plt.show()

## Where to go next

- For batch embedding and the paper’s retrieval, linear-probe, clustering, attribution, and UMAP tasks, start with `uv run python -m instanovo_fm.eval.embed_evaluation --config-name foundational_local`.
- For end-to-end de novo prediction over MGF, mzML, mzXML, CSV, or parquet data, use `instanovo-fm denovo predict` and the options documented in `src/instanovo_fm/downstream/de_novo_sequencing/README.md`.
- The manuscript, [Learning from tandem mass spectra at scale with a self-supervised foundation model for proteomics](https://doi.org/10.64898/2026.09.03.747733), describes the masked-reconstruction objective and the reported use cases in detail.

The most productive next experiment is usually modest: embed a held-out collection from your own workflow, inspect its nearest neighbours and low-dimensional structure, then add the smallest appropriately split probe needed to test a concrete biological or technical hypothesis.